### Imports

In [1]:
from pathlib import Path

import pandas as pd

from src.config import (
    COMBINED_SILVER_PATH,
)

print("Combined silver path:")
print(COMBINED_SILVER_PATH)

Combined silver path:
/media/breezy/NewVolume/projects_int/dengue/data/silver/dengue_weather_weekly.parquet


### Load the data

In [2]:
combined = pd.read_parquet(
    COMBINED_SILVER_PATH
)

print("Shape:", combined.shape)

print("\nColumns:")
for column in combined.columns:
    print(" -", column)

Shape: (26416, 50)

Columns:
 - district
 - year
 - week
 - source_file
 - week_ending
 - epi_week
 - dengue_fever_this_week
 - dengue_fever_cumulative
 - dysentery_this_week
 - dysentery_cumulative
 - encephalitis_this_week
 - encephalitis_cumulative
 - enteric_fever_this_week
 - enteric_fever_cumulative
 - food_poisoning_this_week
 - food_poisoning_cumulative
 - leptospirosis_this_week
 - leptospirosis_cumulative
 - typhus_this_week
 - typhus_cumulative
 - viral_hepatitis_this_week
 - viral_hepatitis_cumulative
 - returns_pct
 - human_rabies_this_week
 - human_rabies_cumulative
 - chickenpox_this_week
 - chickenpox_cumulative
 - meningitis_this_week
 - meningitis_cumulative
 - leishmaniasis_this_week
 - leishmaniasis_cumulative
 - timeliness_pct
 - completeness_pct
 - tuberculosis_this_week
 - tuberculosis_cumulative
 - leprosy_this_week
 - leprosy_cumulative
 - cases
 - week_start
 - is_missing_observation
 - rainfall_mm
 - temperature_mean
 - temperature_max
 - temperature_min
 - h

In [3]:
MODEL_COLUMNS = [
    # Time / geography
    "district",
    "year",
    "week",
    "week_start",

    # Target
    "cases",
    "is_missing_observation",

    # Weather
    "rainfall_mm",
    "temperature_mean",
    "temperature_max",
    "temperature_min",
    "humidity_mean",
    "wind_speed_mean",
    "dtr_mean",

    # Weather coverage / quality
    "weather_days",
    "weather_complete",
    "weather_missing_days",
]

In [4]:
missing_columns = [
    column
    for column in MODEL_COLUMNS
    if column not in combined.columns
]

print("Missing modeling columns:")
print(missing_columns)

Missing modeling columns:
[]


### Create the modeling dataset

In [5]:
modeling = combined[
    MODEL_COLUMNS
].copy()

print("Modeling shape:", modeling.shape)

Modeling shape: (26416, 16)


### Check the target

In [6]:
print("Cases dtype:")
print(modeling["cases"].dtype)

print("\nCases summary:")
print(modeling["cases"].describe())

print("\nMissing cases:")
print(modeling["cases"].isna().sum())

print("\nZero cases:")
print((modeling["cases"] == 0).sum())

Cases dtype:
Int64

Cases summary:
count      26337.0
mean     32.672932
std      82.658857
min            0.0
25%            2.0
50%            9.0
75%           30.0
max         2631.0
Name: cases, dtype: Float64

Missing cases:
79

Zero cases:
3613


### Check missing observations

In [7]:
print(
    modeling[
        "is_missing_observation"
    ].value_counts(dropna=False)
)

is_missing_observation
False    26337
True        79
Name: count, dtype: int64


In [8]:
missing_by_district = (
    modeling
    .groupby("district")["is_missing_observation"]
    .agg(
        total="count",
        missing="sum",
    )
)

missing_by_district["missing_pct"] = (
    missing_by_district["missing"]
    / missing_by_district["total"]
    * 100
)

missing_by_district.sort_values(
    "missing_pct",
    ascending=False,
)

,total,missing,missing_pct
district,,,
Nuwara Eliya,1016,4,0.393701
Ampara,1016,3,0.295276
Badulla,1016,3,0.295276
Batticaloa,1016,3,0.295276
Colombo,1016,3,0.295276
Galle,1016,3,0.295276
Gampaha,1016,3,0.295276
Hambantota,1016,3,0.295276
Jaffna,1016,3,0.295276


### Weather completeness

In [9]:
print(
    modeling[
        [
            "weather_days",
            "weather_complete",
            "weather_missing_days",
        ]
    ].describe()
)

       weather_days  weather_missing_days
count  26416.000000          26416.000000
mean       6.994094              0.005906
std        0.188148              0.188148
min        1.000000              0.000000
25%        7.000000              0.000000
50%        7.000000              0.000000
75%        7.000000              0.000000
max        7.000000              6.000000


In [10]:
incomplete_weather = modeling[
    modeling["weather_complete"] != True
]

print(
    "Incomplete weather rows:",
    len(incomplete_weather)
)

incomplete_weather.head()

Incomplete weather rows: 26


,district,year,week,week_start,cases,is_missing_observation,rainfall_mm,temperature_mean,temperature_max,temperature_min,humidity_mean,wind_speed_mean,dtr_mean,weather_days,weather_complete,weather_missing_days
1015,Ampara,2026,27,2026-06-29,32,False,2.89,28.76,31.23,26.92,78.71,4.68,4.31,1,False,6
2031,Anuradhapura,2026,27,2026-06-29,51,False,0.60,28.04,31.86,25.22,77.52,7.47,6.64,1,False,6
3047,Badulla,2026,27,2026-06-29,115,False,9.75,24.96,28.95,22.33,86.14,4.90,6.62,1,False,6
4063,Batticaloa,2026,27,2026-06-29,52,False,2.89,28.76,31.23,26.92,78.71,4.68,4.31,1,False,6
5079,Colombo,2026,27,2026-06-29,1138,False,13.99,26.79,28.23,25.65,86.88,6.64,2.58,1,False,6


In [11]:
# Target missing
print("Target (cases) missing:")
print(modeling["is_missing_observation"].value_counts(dropna=False))
print("Pct missing:", modeling["is_missing_observation"].mean() * 100)

# Weather missing
print("\nWeather incomplete:")
print((~modeling["weather_complete"]).sum(), "rows")
print("Pct incomplete:", (~modeling["weather_complete"]).mean() * 100)

# Overlap — are they the same rows?
both = modeling["is_missing_observation"] & (~modeling["weather_complete"])
print("\nRows missing BOTH cases and weather:", both.sum())

Target (cases) missing:
is_missing_observation
False    26337
True        79
Name: count, dtype: int64
Pct missing: 0.299061175045427

Weather incomplete:
26 rows
Pct incomplete: 0.09842519685039369

Rows missing BOTH cases and weather: 0


this was  0.3% target missing, 0.1% weather incomplete

### District coverage

In [12]:
print(
    "Number of districts:",
    modeling["district"].nunique()
)

print("\nDistricts:")

for district in sorted(
    modeling["district"].dropna().unique()
):
    print(" -", district)

Number of districts: 26

Districts:
 - Ampara
 - Anuradhapura
 - Badulla
 - Batticaloa
 - Colombo
 - Galle
 - Gampaha
 - Hambantota
 - Jaffna
 - Kalmunai
 - Kalutara
 - Kandy
 - Kegalle
 - Kilinochchi
 - Kurunegala
 - Mannar
 - Matale
 - Matara
 - Monaragala
 - Mullaitivu
 - Nuwara Eliya
 - Polonnaruwa
 - Puttalam
 - Ratnapura
 - Trincomalee
 - Vavuniya


In [13]:
district_counts = (
    modeling
    .groupby("district")
    .size()
    .sort_values()
)

district_counts

district
Ampara          1016
Anuradhapura    1016
Badulla         1016
Batticaloa      1016
Colombo         1016
Galle           1016
Gampaha         1016
Hambantota      1016
Jaffna          1016
Kalmunai        1016
Kalutara        1016
Kandy           1016
Kegalle         1016
Kilinochchi     1016
Kurunegala      1016
Mannar          1016
Matale          1016
Matara          1016
Monaragala      1016
Mullaitivu      1016
Nuwara Eliya    1016
Polonnaruwa     1016
Puttalam        1016
Ratnapura       1016
Trincomalee     1016
Vavuniya        1016
dtype: int64

verify that every canonical district is represented.

### Check duplicate modeling observations

In [14]:
duplicates = modeling[
    modeling.duplicated(
        subset=[
            "district",
            "week_start",
        ],
        keep=False,
    )
]

print(
    "Duplicate rows:",
    len(duplicates)
)

Duplicate rows: 0


### Check chronological ordering

In [15]:
modeling = modeling.sort_values(
    [
        "district",
        "week_start",
    ]
).reset_index(drop=True)


modeling[
    [
        "district",
        "week_start",
        "cases",
    ]
]



,district,week_start,cases
0,Ampara,2007-01-01,0
1,Ampara,2007-01-08,0
2,Ampara,2007-01-15,0
3,Ampara,2007-01-22,0
4,Ampara,2007-01-29,0
...,...,...,...
26411,Vavuniya,2026-06-01,5
26412,Vavuniya,2026-06-08,9
26413,Vavuniya,2026-06-15,114
26414,Vavuniya,2026-06-22,8


### Check weekly gaps

In [16]:
modeling["week_gap_days"] = (
    modeling
    .groupby("district")["week_start"]
    .diff()
    .dt.days
)

print(
    modeling["week_gap_days"]
    .value_counts(dropna=False)
    .sort_index()
)

week_gap_days
7.0     26338
14.0       52
NaN        26
Name: count, dtype: int64


### Check numeric columns

In [17]:
numeric_columns = [
    "cases",
    "rainfall_mm",
    "temperature_mean",
    "temperature_max",
    "temperature_min",
    "humidity_mean",
    "wind_speed_mean",
    "dtr_mean",
    "weather_days",
    "weather_missing_days",
]

modeling[numeric_columns].dtypes

cases                     Int64
rainfall_mm             float64
temperature_mean        float64
temperature_max         float64
temperature_min         float64
humidity_mean           float64
wind_speed_mean         float64
dtr_mean                float64
weather_days              int64
weather_missing_days      int64
dtype: object

### Check impossible values

In [18]:
checks = {
    "negative cases": (
        modeling["cases"] < 0
    ).sum(),

    "negative rainfall": (
        modeling["rainfall_mm"] < 0
    ).sum(),

    "negative humidity": (
        modeling["humidity_mean"] < 0
    ).sum(),

    "humidity > 100": (
        modeling["humidity_mean"] > 100
    ).sum(),

    "negative weather days": (
        modeling["weather_days"] < 0
    ).sum(),

    "negative missing days": (
        modeling["weather_missing_days"] < 0
    ).sum(),
}

for name, count in checks.items():
    print(
        f"{name}: {count}"
    )

negative cases: 0
negative rainfall: 0
negative humidity: 0
humidity > 100: 0
negative weather days: 0
negative missing days: 0


### Check target leakage

In [19]:
leakage_columns = [
    column
    for column in modeling.columns
    if "dengue" in column.lower()
]

print("Dengue-related columns:")
print(leakage_columns)

Dengue-related columns:
[]


### Save the modeling layer

In [20]:
from src.config import MODEL_DATA_PATH

MODEL_DATA_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)

modeling.to_parquet(
    MODEL_DATA_PATH,
    index=False,
)

print(
    "Saved:",
    MODEL_DATA_PATH
)

Saved: /media/breezy/NewVolume/projects_int/dengue/data/gold/dengue_modeling.parquet


In [22]:
### Reload test
test_reload = pd.read_parquet(
    MODEL_DATA_PATH
)

print(
    "Reloaded shape:",
    test_reload.shape
)

print(
    "Columns:",
    list(test_reload.columns)
)

Reloaded shape: (26416, 17)
Columns: ['district', 'year', 'week', 'week_start', 'cases', 'is_missing_observation', 'rainfall_mm', 'temperature_mean', 'temperature_max', 'temperature_min', 'humidity_mean', 'wind_speed_mean', 'dtr_mean', 'weather_days', 'weather_complete', 'weather_missing_days', 'week_gap_days']


In [24]:
assert test_reload.shape == modeling.shape

assert list(
    test_reload.columns
) == list(
    modeling.columns
)

print(
    "Modeling dataset save/reload test passed"
)

Modeling dataset save/reload test passed


### Final dataset audit

In [25]:
print("=" * 70)
print("MODEL DATASET AUDIT")
print("=" * 70)

print(
    f"Rows:       {len(modeling):,}"
)

print(
    f"Columns:    {len(modeling.columns)}"
)

print(
    f"Districts:  {modeling['district'].nunique()}"
)

print(
    f"Start:      {modeling['week_start'].min()}"
)

print(
    f"End:        {modeling['week_start'].max()}"
)

print(
    f"Duplicates: {modeling.duplicated(['district', 'week_start']).sum()}"
)

print(
    f"Missing target: {modeling['cases'].isna().sum():,}"
)

print(
    f"Missing weather rows: "
    f"{modeling['weather_complete'].ne(True).sum():,}"
)

MODEL DATASET AUDIT
Rows:       26,416
Columns:    17
Districts:  26
Start:      2007-01-01 00:00:00
End:        2026-06-29 00:00:00
Duplicates: 0
Missing target: 79
Missing weather rows: 26
